### Realtime shift checks


In [13]:
import numpy  as np
import astropy.units as u
import re

In [14]:
def parse_alert_info(text: str) -> dict:
    # Match pattern: Value + PlusError - MinusError
    pattern = r"(RA|Dec)\s*=\s*([0-9.]+)\s*\+\s*([0-9.]+)\s*-\s*([0-9.]+)"

    # Find all matches (Expects 4 matches: 50% RA, 50% Dec, 90% RA, 90% Dec)
    matches = re.findall(pattern, text)

    if len(matches) < 4:
        raise ValueError("Could not find all required RA/Dec values in text.")

    # Extract match components
    ra_50, dec_50, ra_90, dec_90 = matches

    return {
        "RA": float(ra_50[1]),
        "DEC": float(dec_50[1]),
        "RA_err_50_plus": float(ra_50[2]),
        "RA_err_50_minus": float(ra_50[3]),
        "RA_err_90_plus": float(ra_90[2]),
        "RA_err_90_minus": float(ra_90[3]),
        "DEC_err_50_plus": float(dec_50[2]),
        "DEC_err_50_minus": float(dec_50[3]),
        "DEC_err_90_plus": float(dec_90[2]),
        "DEC_err_90_minus": float(dec_90[3]),
    }

In [15]:
alert_info_message = '''
Approximating the 50% error region as a rectangle, we get (results in degrees): 
     RA = 224.12 + 0.31 - 0.30 
     Dec = 20.62 + 0.33 - 0.33 
 
Approximating the 90% error region as a rectangle, we get (results in degrees): 
     RA = 224.12 + 0.56 - 0.56 
     Dec = 20.62 + 0.61 - 0.61 
 
Contour area at 50% = 0.2900 sqdeg 
 
Contour area at 90% = 1.0000 sqdeg
'''

In [16]:
alert = parse_alert_info(alert_info_message)

print("Alert info from message:\n")
print("Error 50%")
print(f"RA = {alert['RA']} + {alert['RA_err_50_plus']} - {alert['RA_err_50_minus']}")
print(f"DEC = {alert['DEC']} + {alert['DEC_err_50_plus']} - {alert['DEC_err_50_minus']}\n")

print("Error 90%")
print(f"RA = {alert['RA']} + {alert['RA_err_90_plus']} - {alert['RA_err_90_minus']}")
print(f"DEC = {alert['DEC']} + {alert['DEC_err_90_plus']} - {alert['DEC_err_90_minus']}")

Alert info from message:

Error 50%
RA = 224.12 + 0.31 - 0.3
DEC = 20.62 + 0.33 - 0.33

Error 90%
RA = 224.12 + 0.56 - 0.56
DEC = 20.62 + 0.61 - 0.61


In [17]:
dec = alert['DEC']

ra_err_plus_50 = alert['RA_err_50_plus']
ra_err_minus_50 = alert['RA_err_50_minus']
dec_err_plus_50 = alert['DEC_err_50_plus']
dec_err_minus_50 = alert['DEC_err_50_minus']


ra_err_plus_90 = alert['RA_err_90_plus']
ra_err_minus_90 = alert['RA_err_90_minus']
dec_err_plus_90 = alert['DEC_err_90_plus']
dec_err_minus_90 = alert['DEC_err_90_minus']

In [19]:

#50%
ra_err_mean_50 = (abs(ra_err_plus_50) + abs(ra_err_minus_50)) / 2
dec_err_mean_50 = (abs(dec_err_plus_50) + abs(dec_err_minus_50)) / 2
err_combined_50 = np.sqrt( ra_err_mean_50 * np.cos(np.deg2rad(dec)) * dec_err_mean_50 )
print("Error combined 50%", err_combined_50)


ra_err_mean_90 = (abs(ra_err_plus_90) + abs(ra_err_minus_90)) / 2
dec_err_mean_90 = (abs(dec_err_plus_90) + abs(dec_err_minus_90)) / 2
err_combined_90 = np.sqrt(ra_err_mean_90 * np.cos(np.deg2rad(dec)) * dec_err_mean_90 )
print("Error combined 90%", err_combined_90)

Error combined 50% 0.30692348413204124
Error combined 90% 0.5654343145013762


In [7]:
ra = alert['RA'] * u.deg
dec = alert['DEC'] * u.deg
print("Ra, dec: [", ra.to(u.rad), ',', dec.to(u.rad), ']')

Ra, dec: [ 3.9116319195696914 rad , 0.3598868917612308 rad ]
